# Text-to-Video (TTV) — video generation benchmark on Apple Silicon

Generates one video per prompt with each model in **float16** on MLX and records, per prompt: resolution, frames, video
length, model load, text encoding, denoising, time per step, VAE decode, inference time, inference rate and peak memory,
plus system metrics (CPU / GPU / NPU utilisation, power, temperature) sampled by `macmon`.

| Model | MLX library | Runs in |
|---|---|---|
| Wan2.1-T2V-1.3B, Wan2.2-TI2V-5B | [`mlx-video`](https://github.com/Blaizzy/mlx-video) | this notebook |
| CogVideoX-Fun-V1.5-5b-InP | [`VideoX-Fun-mlx`](https://github.com/dgrauet/VideoX-Fun-mlx) (in `ttv_backends/`) | this notebook |
| Lance-3B-Video | [`lance-mlx`](https://github.com/xocialize/lance-mlx) (in `ttv_backends/`) | `.venv-lance` (it pins its own `mlx-vlm` / `mlx-video`), via `ttv_backends/lance_t2v.py` |

Each model is prepared once as a local float16 copy at `models/<model>-fp16/`: the Wan models are converted from their
original checkpoints (VAE kept float32, as the converter requires); the CogVideoX and Lance MLX weights have their
bfloat16 tensors cast to float16. Every library reloads its model for each prompt, so inference time includes loading.

**Outputs**
- `outputs/ttv/<family>/<params>/<prompt set>/`: one `.mp4` per prompt and `<model>_simple_metrics.json`
- `metrics_analysis/ttv/<family>/<params>/<prompt set>/`: `Model_Level_Metrics/<model>_prompt_metrics.csv`
  and `System_Level_Metrics/<prompt id>_system_metrics.csv`

## 1. Settings and model registry

In [1]:
import contextlib
import io
import json
import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

import cv2
import imageio.v3 as iio
import mlx.core as mx
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'prompts').is_dir())   # project root
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'ttv_backends' / 'VideoX-Fun-mlx'))
os.environ.setdefault('HF_HUB_DISABLE_IMPLICIT_TOKEN', '1')   # every model here is public

# family          size      params  model                        source on Hugging Face                     params (B)  library
REGISTRY = [
    ('Wan2.1',        'SMALL',  '1.3B', 'Wan2.1-T2V-1.3B',           'Wan-AI/Wan2.1-T2V-1.3B',                  1.42, 'mlx-video'),
    ('Wan2.2',        'SMALL',  '5B',   'Wan2.2-TI2V-5B',            'Wan-AI/Wan2.2-TI2V-5B',                   5.0,  'mlx-video'),
    ('CogVideoX-Fun', 'LARGE',  '5B',   'CogVideoX-Fun-V1.5-5b-InP', 'dgrauet/CogVideoX-Fun-V1.5-5b-InP-mlx',   5.57, 'videox-fun-mlx'),
    ('Lance',         'SMALL',  '3B',   'Lance-3B-Video',            'mlx-community/Lance-3B-Video-bf16',       6.44, 'lance-mlx'),
]
MODELS = pd.DataFrame(REGISTRY, columns=['family', 'size', 'params', 'model', 'source', 'params_b', 'library']).set_index('model')
MODELS['path'] = 'models/' + MODELS.index + '-fp16'   # local float16 copy of each model

RUN = os.environ.get('TTV_MODELS', ','.join(MODELS.index)).split(',')   # models to run, in order
PROMPT_SET = os.environ.get('TTV_PROMPTS', 'ttv_prompts_10')            # prompts/ttv_prompts/<set>.csv (id, prompt)

# Generation settings, identical for every model and prompt (quick settings, well below each model's official ones)
WIDTH = int(os.environ.get('TTV_WIDTH', 480))
HEIGHT = int(os.environ.get('TTV_HEIGHT', 480))
FRAMES = int(os.environ.get('TTV_FRAMES', 33))     # must be 4n + 1 (CogVideoX needs an even latent count: 33 -> 37)
STEPS = int(os.environ.get('TTV_STEPS', 20))
GUIDANCE = float(os.environ.get('TTV_GUIDANCE', 5.0))
FPS = int(os.environ.get('TTV_FPS', 16))           # playback rate every video is saved at
SEED = 0

print(f'Models: {RUN}  |  {WIDTH} x {HEIGHT}, {FRAMES} frames, {STEPS} steps, guidance {GUIDANCE}, {FPS} fps, seed {SEED}')
MODELS[['family', 'size', 'params', 'library', 'source', 'path']]

Models: ['Wan2.1-T2V-1.3B', 'Wan2.2-TI2V-5B', 'CogVideoX-Fun-V1.5-5b-InP', 'Lance-3B-Video']  |  480 x 480, 33 frames, 20 steps, guidance 5.0, 16 fps, seed 0


,family,size,params,library,source,path
model,,,,,,
Wan2.1-T2V-1.3B,Wan2.1,SMALL,1.3B,mlx-video,Wan-AI/Wan2.1-T2V-1.3B,models/Wan2.1-T2V-1.3B-fp16
Wan2.2-TI2V-5B,Wan2.2,SMALL,5B,mlx-video,Wan-AI/Wan2.2-TI2V-5B,models/Wan2.2-TI2V-5B-fp16
CogVideoX-Fun-V1.5-5b-InP,CogVideoX-Fun,LARGE,5B,videox-fun-mlx,dgrauet/CogVideoX-Fun-V1.5-5b-InP-mlx,models/CogVideoX-Fun-V1.5-5b-InP-fp16
Lance-3B-Video,Lance,SMALL,3B,lance-mlx,mlx-community/Lance-3B-Video-bf16,models/Lance-3B-Video-fp16


## 2. Prepare the models

The first time, each model is downloaded and saved as a float16 copy in `models/<model>-fp16/` (one-time step).

In [2]:
from huggingface_hub import hf_hub_download, snapshot_download
from mlx_video.models.wan_2.convert import convert_wan_checkpoint

def fp16_copy(src, dst):
    # Copy an MLX model folder, casting its bfloat16 tensors to float16
    dst.mkdir(parents=True)
    for f in Path(src).iterdir():
        if f.suffix == '.safetensors':
            weights, meta = mx.load(str(f), return_metadata=True)
            mx.save_safetensors(str(dst / f.name), {k: v.astype(mx.float16) if v.dtype == mx.bfloat16 else v
                                                    for k, v in weights.items()}, metadata=meta)
        elif f.is_file():
            shutil.copyfile(f, dst / f.name)

def prepare(name):
    m = MODELS.loc[name]
    path = Path(m.path)
    if not path.exists():
        print(f'Preparing {m.path} from {m.source} (float16), one-time step...')
        src = snapshot_download(m.source, token=False, allow_patterns=['*.json', '*.pth', '*.safetensors', '*.model', '*.txt'])
        try:
            if m.library == 'mlx-video':
                convert_wan_checkpoint(src, m.path, dtype='float16')
            else:
                fp16_copy(src, path)
        except Exception:
            shutil.rmtree(path, ignore_errors=True)   # don't leave a half-prepared model behind
            raise
    if m.library == 'videox-fun-mlx' and not (path / 'tokenizer_spiece.model').exists():
        # The MLX repo lacks the T5 tokenizer model: take it from the original checkpoint
        spiece = hf_hub_download('alibaba-pai/CogVideoX-Fun-V1.5-5b-InP', 'tokenizer/spiece.model', token=False)
        shutil.copyfile(spiece, path / 'tokenizer_spiece.model')

for name in RUN:
    prepare(name)
pd.Series({name: f"{sum(f.stat().st_size for f in Path(MODELS.path[name]).glob('*.safetensors')) / 1e9:.1f} GB"
           for name in RUN}, name='float16 weights on disk')

/Users/genailab/mlx-projects/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Wan2.1-T2V-1.3B              14.7 GB
Wan2.2-TI2V-5B               24.2 GB
CogVideoX-Fun-V1.5-5b-InP    21.1 GB
Lance-3B-Video               15.6 GB
Name: float16 weights on disk, dtype: str

## 3. Prompts

In [3]:
PROMPTS = pd.read_csv(Path('prompts/ttv_prompts') / f'{PROMPT_SET}.csv', dtype={'id': str})
PROMPTS['id'] = PROMPTS.id.str.zfill(2)
print(f'{len(PROMPTS)} prompts from {PROMPT_SET}.csv')
PROMPTS

10 prompts from ttv_prompts_10.csv


,id,prompt
0,01,3D render of a cute tropical fish in an aquari...
1,02,An armchair in the shape of an avocado
2,03,A van Gogh style painting of an American footb...
3,04,A pencil and watercolor drawing of a bright ci...
4,05,An abstract visual of artificial intelligence
5,06,A Fat Cat playing Golf
6,07,A hand-drawn sailboat circled by birds on the ...
7,08,A 3D render of a rainbow colored hot air ballo...
8,09,A Formula 1 car driving on a neon road
9,10,illustration of a dinosaur enjoying a peaceful...


## 4. Generate the videos

One function per library generates a video and returns its stage times. Each prompt is wrapped in
`metrics_monitor_start()` / `metrics_monitor_stop()` (from `macmon_metrics/`), which saves that prompt's system metrics
every 100 ms. A fixed seed makes every run repeatable.

In [4]:
import macmon_metrics.metrics_monitor as mm
from macmon_metrics.metrics_monitor import metrics_monitor_start, metrics_monitor_stop, setup_output_directory
from mlx_video.models.wan_2.generate import generate_video

if mm._tracker is None:   # start macmon now, so every prompt's capture has samples from its first moment
    mm._tracker = mm.MacMonTracker(binary_path=mm._resolve_macmon_binary())
    mm._tracker.start()
    time.sleep(1.5)

def wan_generate(m, prompt, out):
    # mlx-video: loads the text encoder, encodes, loads the transformer, denoises, decodes; reports each stage
    log = io.StringIO()
    with contextlib.redirect_stdout(log):
        generate_video(model_dir=m.path, prompt=prompt, width=WIDTH, height=HEIGHT, num_frames=FRAMES,
                       steps=STEPS, guide_scale=GUIDANCE, seed=SEED, output_path=str(out))
    text = re.sub(r'\x1b\[[0-9;]*m', '', log.getvalue())
    stage = lambda key: float(re.search(rf'{key}: ([\d.]+)s', text).group(1))
    return {'model_load (s)': stage('Models loaded'), 'text_encoding (s)': stage('T5 encoding'),   # T5 load + encode
            'denoising (s)': stage('Denoising'), 'vae_decode (s)': stage('VAE decode')}

class ZeroLatents:
    # Stands in for the VAE encoding of the blank input clip: text-to-video starts from noise, as in VideoX-Fun's own
    # text-to-video path, and encoding a blank 480 x 480 clip would only cost memory
    def __init__(self, x):
        self.shape = (x.shape[0], (x.shape[1] - 1) // 4 + 1, x.shape[2] // 8, x.shape[3] // 8, 16)

    def mode(self):
        return mx.zeros(self.shape)

def cogvideox_generate(m, prompt, out):
    import gc
    from videox_fun_mlx.pipeline.pipeline_cogvideox_fun_inpaint import CogVideoXFunInpaintPipeline
    t0 = time.perf_counter()
    with contextlib.redirect_stdout(io.StringIO()):
        pipe = CogVideoXFunInpaintPipeline.from_pretrained(m.path)
    mx.eval(pipe.transformer.parameters(), pipe.text_encoder.parameters(), pipe.vae.parameters())
    t1 = time.perf_counter()
    embeds = pipe.encode_prompt(prompt, '', do_cfg=True)
    mx.eval(embeds)
    pipe.text_encoder = None   # prompt encoded: free the 9.5 GB T5 encoder
    t2 = time.perf_counter()
    decode = {}

    def framewise_decode(z):
        # Decode 2 latent frames at a time, carrying the causal conv cache (as diffusers' CogVideoX VAE does);
        # decoding the whole clip at once needs far more memory than this Mac has
        pipe.transformer = None   # denoising finished: free the transformer first
        gc.collect()
        mx.clear_cache()
        d0 = time.perf_counter()
        n, rem = z.shape[1], z.shape[1] % 2
        cache, parts = None, []
        for i in range(max(n // 2, 1)):
            part, cache = pipe.vae.decoder(z[:, 2 * i + (0 if i == 0 else rem): 2 * (i + 1) + rem], conv_cache=cache)
            mx.eval(part, cache)
            parts.append(part)
        decode['s'] = time.perf_counter() - d0
        return mx.concatenate(parts, axis=1)

    pipe.vae.encode, pipe.vae.decode = ZeroLatents, framewise_decode
    # CogVideoX-Fun V1.5 pairs latent frames, so it needs an even latent count: 33 frames -> 37 (10 latent frames)
    frames = FRAMES + 4 if ((FRAMES - 1) // 4 + 1) % 2 else FRAMES
    # Text-to-video = inpainting a fully masked blank clip (how CogVideoX-Fun InP generates from text only)
    video = pipe(video=mx.zeros((1, frames, HEIGHT, WIDTH, 3)), mask=mx.ones((1, frames, HEIGHT, WIDTH, 1)),
                 prompt_embeds=embeds, num_inference_steps=STEPS, guidance_scale=GUIDANCE, seed=SEED)
    mx.eval(video)
    t3 = time.perf_counter()
    iio.imwrite(str(out), (np.array(video[0].astype(mx.float32)) * 255).clip(0, 255).astype(np.uint8),
                fps=FPS, codec='libx264', macro_block_size=1)
    del pipe, video
    gc.collect()
    mx.clear_cache()
    return {'model_load (s)': round(t1 - t0, 1), 'text_encoding (s)': round(t2 - t1, 1),
            'denoising (s)': round(t3 - t2 - decode['s'], 1), 'vae_decode (s)': round(decode['s'], 1)}

def lance_generate(m, prompt, out):
    # Runs in .venv-lance; the driver prints its stage times (and its own peak memory) as JSON
    run = subprocess.run(['.venv-lance/bin/python', 'ttv_backends/lance_t2v.py', '--weights', m.path, '--prompt', prompt,
                          '--output', str(out), '--width', str(WIDTH), '--height', str(HEIGHT), '--frames', str(FRAMES),
                          '--steps', str(STEPS), '--cfg', str(GUIDANCE), '--seed', str(SEED), '--fps', str(FPS)],
                         capture_output=True, text=True)
    if run.returncode != 0:
        raise RuntimeError(run.stderr[-2000:])
    return json.loads(run.stdout.strip().splitlines()[-1])

GENERATE = {'mlx-video': wan_generate, 'videox-fun-mlx': cogvideox_generate, 'lance-mlx': lance_generate}

def run_model(name):
    # Generate one video per prompt; save videos, per-prompt metrics and a summary
    m = MODELS.loc[name]
    sub = Path(m.family) / m.params / PROMPT_SET
    out_dir = Path('outputs/ttv') / sub
    out_dir.mkdir(parents=True, exist_ok=True)
    sys_dir, model_dir = setup_output_directory(str(Path('metrics_analysis/ttv') / sub))
    rows = []
    for p in PROMPTS.itertuples():
        video = out_dir / f'{name}_video_{p.id}.mp4'
        mx.reset_peak_memory()
        metrics_monitor_start(idx=p.id, metrics_folder=sys_dir)
        t0 = time.perf_counter()
        stages = GENERATE[m.library](m, p.prompt, video)
        total = time.perf_counter() - t0
        metrics_monitor_stop()
        peak = stages.pop('peak_memory (GB)', round(mx.get_peak_memory() / 1e9, 2))
        cap = cv2.VideoCapture(str(video))
        if round(cap.get(cv2.CAP_PROP_FPS)) != FPS:   # mlx-video saves at the model's own rate (Wan2.2: 24 fps)
            cap.release()
            iio.imwrite(str(video), iio.imread(str(video)), fps=FPS, codec='libx264', macro_block_size=1)
            cap = cv2.VideoCapture(str(video))
        n_frames, fps = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), cap.get(cv2.CAP_PROP_FPS)
        w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()
        rows.append({
            'prompt_id': p.id, 'model': name, 'family': m.family, 'size': m['size'],
            'resolution (px)': f'{w}x{h}', 'frames': n_frames, 'fps': round(fps), 'video_duration (s)': round(n_frames / fps, 2),
            'steps': STEPS, 'guidance': GUIDANCE, 'seed': SEED,
            'input_words': len(p.prompt.split()), 'input_characters': len(p.prompt),
            **stages,
            'time_per_step (s)': round(stages['denoising (s)'] / STEPS, 3),
            'inference_time (s)': round(total, 2),
            'inference_rate (videos/min)': round(60 / total, 3),
            'peak_memory (GB)': peak,
            'video_file': str(video), 'prompt': p.prompt,
        })
        print(f"  [{p.id}] {total:.1f} s (denoising {stages['denoising (s)']} s), peak {peak} GB -> {video}")

    df = pd.DataFrame(rows)
    df.to_csv(Path(model_dir) / f'{name}_prompt_metrics.csv', index=False)
    hw = pd.concat([pd.read_csv(Path(sys_dir) / f'{i}_system_metrics.csv') for i in df.prompt_id])
    summary = {
        'model': name, 'family': m.family, 'size': m['size'], 'parameters': m.params, 'precision': 'float16',
        'library': m.library, 'path': m.path, 'source': m.source, 'prompt_set': PROMPT_SET, 'videos': len(df),
        'resolution (px)': f'{WIDTH}x{HEIGHT}', 'frames': int(df.frames.iloc[0]), 'steps': STEPS, 'guidance': GUIDANCE, 'fps': FPS, 'seed': SEED,
        'model_load (s, avg)': round(df['model_load (s)'].mean(), 2),
        'text_encoding (s, avg)': round(df['text_encoding (s)'].mean(), 2),
        'denoising (s, avg)': round(df['denoising (s)'].mean(), 2),
        'time_per_step (s, avg)': round(df['time_per_step (s)'].mean(), 3),
        'vae_decode (s, avg)': round(df['vae_decode (s)'].mean(), 2),
        'inference_time (s, avg)': round(df['inference_time (s)'].mean(), 2),
        'inference_time (s, total)': round(df['inference_time (s)'].sum(), 2),
        'inference_rate (videos/min, avg)': round(df['inference_rate (videos/min)'].mean(), 3),
        'peak_memory (GB)': df['peak_memory (GB)'].max(),
        'gpu_utilization (%, avg)': round(hw['GPU Utilization (%)'].mean(), 1),
        'gpu_power (W, avg)': round(hw['gpuPower'].mean(), 1),
        'gpu_temperature (C, max)': round(hw['GPU_Temperature(C)'].max(), 1),
        'cpu_utilization (%, avg)': round(hw['CPU_Util(%)'].mean(), 1),
        'cpu_power (W, avg)': round(hw['cpuPower'].mean(), 1),
        'npu_power (W, avg)': round(hw['npuPower'].mean(), 2),
        'system_energy (J)': round(hw['total_system_power'].sum() * 0.1, 1),   # 100 ms samples
    }
    (out_dir / f'{name}_simple_metrics.json').write_text(json.dumps(summary, indent=2))
    return summary

results = {}
for name in RUN:
    print(name)
    results[name] = run_model(name)
pd.DataFrame(results)

Wan2.1-T2V-1.3B


Diffusion:   0%|          | 0/20 [00:00<?, ?it/s]

Diffusion:   5%|▌         | 1/20 [00:01<00:28,  1.50s/it]

Diffusion:  10%|█         | 2/20 [00:02<00:26,  1.49s/it]

Diffusion:  15%|█▌        | 3/20 [00:04<00:25,  1.48s/it]

...

  [07] 133.7 s (denoising 96.8 s), peak 24.61 GB -> outputs/ttv/Lance/3B/ttv_prompts_10/Lance-3B-Video_video_07.mp4


  [08] 134.5 s (denoising 97.5 s), peak 24.61 GB -> outputs/ttv/Lance/3B/ttv_prompts_10/Lance-3B-Video_video_08.mp4


  [09] 134.8 s (denoising 97.4 s), peak 24.61 GB -> outputs/ttv/Lance/3B/ttv_prompts_10/Lance-3B-Video_video_09.mp4


  [10] 134.0 s (denoising 97.0 s), peak 24.61 GB -> outputs/ttv/Lance/3B/ttv_prompts_10/Lance-3B-Video_video_10.mp4


,Wan2.1-T2V-1.3B,Wan2.2-TI2V-5B,CogVideoX-Fun-V1.5-5b-InP,Lance-3B-Video
model,Wan2.1-T2V-1.3B,Wan2.2-TI2V-5B,CogVideoX-Fun-V1.5-5b-InP,Lance-3B-Video
family,Wan2.1,Wan2.2,CogVideoX-Fun,Lance
size,SMALL,SMALL,LARGE,SMALL
parameters,1.3B,5B,5B,3B
precision,float16,float16,float16,float16
library,mlx-video,mlx-video,videox-fun-mlx,lance-mlx
path,models/Wan2.1-T2V-1.3B-fp16,models/Wan2.2-TI2V-5B-fp16,models/CogVideoX-Fun-V1.5-5b-InP-fp16,models/Lance-3B-Video-fp16
source,Wan-AI/Wan2.1-T2V-1.3B,Wan-AI/Wan2.2-TI2V-5B,dgrauet/CogVideoX-Fun-V1.5-5b-InP-mlx,mlx-community/Lance-3B-Video-bf16
prompt_set,ttv_prompts_10,ttv_prompts_10,ttv_prompts_10,ttv_prompts_10
videos,10,10,10,10


## 5. All models run so far

Combines every float16 model run on `PROMPT_SET` into `outputs/ttv/all_models_<prompt set>.csv` and their per-prompt
metrics into `metrics_analysis/ttv/all_models_prompt_metrics_<prompt set>.csv`.

In [5]:
summaries = [json.loads(f.read_text()) for f in Path('outputs/ttv').glob(f'*/*/{PROMPT_SET}/*_simple_metrics.json')]
summaries = sorted((s for s in summaries if s.get('precision') == 'float16'), key=lambda s: list(MODELS.index).index(s['model']))
all_models = pd.DataFrame(summaries)
all_models.to_csv(Path('outputs/ttv') / f'all_models_{PROMPT_SET}.csv', index=False)
all_prompts = pd.concat([
    pd.read_csv(Path('metrics_analysis/ttv') / s['family'] / s['parameters'] / PROMPT_SET / 'Model_Level_Metrics' /
                f"{s['model']}_prompt_metrics.csv", dtype={'prompt_id': str})
    for s in summaries])
all_prompts.to_csv(Path('metrics_analysis/ttv') / f'all_models_prompt_metrics_{PROMPT_SET}.csv', index=False)
print(f'{len(all_models)} models, {len(all_prompts)} prompt rows')
all_models

4 models, 40 prompt rows


,model,family,size,parameters,precision,library,path,source,prompt_set,videos,...,"inference_time (s, total)","inference_rate (videos/min, avg)",peak_memory (GB),"gpu_utilization (%, avg)","gpu_power (W, avg)","gpu_temperature (C, max)","cpu_utilization (%, avg)","cpu_power (W, avg)","npu_power (W, avg)",system_energy (J)
0,Wan2.1-T2V-1.3B,Wan2.1,SMALL,1.3B,float16,mlx-video,models/Wan2.1-T2V-1.3B-fp16,Wan-AI/Wan2.1-T2V-1.3B,ttv_prompts_10,10,...,495.15,1.214,49.39,70.6,43.8,96.8,12.6,3.8,0.0,52020.8
1,Wan2.2-TI2V-5B,Wan2.2,SMALL,5B,float16,mlx-video,models/Wan2.2-TI2V-5B-fp16,Wan-AI/Wan2.2-TI2V-5B,ttv_prompts_10,10,...,368.49,1.631,35.60,63.9,43.0,91.0,19.4,5.2,0.0,37349.9
2,CogVideoX-Fun-V1.5-5b-InP,CogVideoX-Fun,LARGE,5B,float16,videox-fun-mlx,models/CogVideoX-Fun-V1.5-5b-InP-fp16,dgrauet/CogVideoX-Fun-V1.5-5b-InP-mlx,ttv_prompts_10,10,...,1522.83,0.394,55.28,58.8,28.9,91.4,22.3,4.4,0.0,128273.0
3,Lance-3B-Video,Lance,SMALL,3B,float16,lance-mlx,models/Lance-3B-Video-fp16,mlx-community/Lance-3B-Video-bf16,ttv_prompts_10,10,...,1338.35,0.448,24.61,74.7,37.2,85.1,5.5,2.9,0.0,130115.7
